# Notebook 05b — Reentrenamiento con KITTI + Ilusiones Visuales

**Proyecto:** Modelo Cognitivo Artificial para la Replicación de la Actividad Neurofisiológica de Percepción de Profundidad  
**Autor:** Jesús Goenaga Peña  

---

## ¿Qué hace este notebook?

Reentrena el modelo cognitivo artificial incorporando el dataset de ilusiones visuales
(3D Visual Illusion Depth Estimation, Yao et al., NeurIPS 2025) junto con KITTI.

**Motivación:** El modelo original fue entrenado solo con KITTI, lo que limitó su desempeño
en tareas de ilusiones visuales (11% de accuracy). Al incluir ilusiones en el entrenamiento,
el modelo aprende a replicar la percepción humana ante estímulos ambiguos.

**Etiquetado de ilusiones:** Las etiquetas de profundidad relativa se generaron con:
- Video10 (Anamorphic): scale_factors.csv del dataset original (89,642 registros)
- Video0/Video1: varianza residual validada manualmente (15/15 imágenes confirmadas)

**Parámetros de entrenamiento** (idénticos al NB05 original):
- Etapa 1: 30 épocas, NGL+V1 congelados, lr=0.001
- Etapa 2: 70 épocas, fine-tuning completo, lr=0.0001
- Loss: BCELoss | Optimizer: Adam | Scheduler: ReduceLROnPlateau | Patience: 15

---

## Celda 1 — Montar Drive y configurar entorno

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, time
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torchvision.transforms as transforms
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

BASE_DIR        = '/content/drive/MyDrive/cognitive-depth-model'
KITTI_LEFT_DIR  = os.path.join(BASE_DIR, 'data/raw/kitti/data_scene_flow/training/image_2')
KITTI_RIGHT_DIR = os.path.join(BASE_DIR, 'data/raw/kitti/data_scene_flow/training/image_3')
KITTI_DISP_DIR  = os.path.join(BASE_DIR, 'data/raw/kitti/data_scene_flow/training/disp_occ_0')
ILLUSION_DIR    = os.path.join(BASE_DIR, 'data/raw/illusions/images')
ILLUSION_META   = os.path.join(BASE_DIR, 'data/raw/illusions/illusion_metadata_labeled.csv')
SPLITS_DIR      = os.path.join(BASE_DIR, 'data/splits')
CHECKPOINT_DIR  = os.path.join(BASE_DIR, 'checkpoints')
RESULTS_DIR     = os.path.join(BASE_DIR, 'results')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print()
print('Verificación de rutas:')
for nombre, ruta in [
    ('KITTI imágenes izq', KITTI_LEFT_DIR),
    ('KITTI imágenes der', KITTI_RIGHT_DIR),
    ('KITTI disparidad',   KITTI_DISP_DIR),
    ('Ilusiones imágenes', ILLUSION_DIR),
    ('Ilusiones metadata', ILLUSION_META),
]:
    existe = os.path.exists(ruta)
    print(f'  {"✓" if existe else "✗"}  {nombre}')

## Celda 2 — Dataset KITTI

In [ ]:
class KITTIDataset(Dataset):
    """
    Dataset KITTI Scene Flow 2015.
    Entrada: par estereoscópico (imagen izq + imagen der) → 6 canales.
    Etiqueta: 1 si disparidad media >= mediana (objeto cercano), 0 si no.
    """
    def __init__(self, scene_ids, left_dir, right_dir, disp_dir,
                 target_size=(192, 640), augment=False):
        self.left_dir    = left_dir
        self.right_dir   = right_dir
        self.disp_dir    = disp_dir
        self.target_size = target_size
        self.augment     = augment
        self.mean = np.array([0.485,0.456,0.406,0.485,0.456,0.406], dtype=np.float32)
        self.std  = np.array([0.229,0.224,0.225,0.229,0.224,0.225], dtype=np.float32)

        # Filtrar escenas que existen y calcular etiquetas
        self.escenas = []
        disparidades = []
        for sid in scene_ids:
            ruta_izq  = os.path.join(left_dir,  f'{sid}.png')
            ruta_disp = os.path.join(disp_dir,   f'{sid}.png')
            if os.path.exists(ruta_izq) and os.path.exists(ruta_disp):
                mapa = cv2.imread(ruta_disp, cv2.IMREAD_UNCHANGED)
                if mapa is not None:
                    mapa_f  = mapa.astype(np.float32) / 256.0
                    validos = mapa_f[mapa_f > 0]
                    if len(validos) > 50:
                        disp_media = float(np.mean(validos))
                        disparidades.append(disp_media)
                        self.escenas.append((sid, disp_media))

        # Umbral = mediana
        self.umbral = float(np.median(disparidades)) if disparidades else 30.0
        print(f'KITTIDataset: {len(self.escenas)} escenas | umbral disp={self.umbral:.2f}px')

    def __len__(self):
        return len(self.escenas)

    def __getitem__(self, idx):
        sid, disp_media = self.escenas[idx]
        label = 1.0 if disp_media >= self.umbral else 0.0

        ruta_izq = os.path.join(self.left_dir,  f'{sid}.png')
        ruta_der = os.path.join(self.right_dir,  f'{sid}.png')

        img_izq = cv2.imread(ruta_izq)
        img_der = cv2.imread(ruta_der) if os.path.exists(ruta_der) else img_izq
        if img_izq is None: img_izq = np.zeros((375,1242,3), dtype=np.uint8)
        if img_der is None: img_der = img_izq.copy()

        h, w = self.target_size
        img_izq = cv2.resize(cv2.cvtColor(img_izq, cv2.COLOR_BGR2RGB), (w, h))
        img_der = cv2.resize(cv2.cvtColor(img_der, cv2.COLOR_BGR2RGB), (w, h))

        if self.augment and np.random.rand() > 0.5:
            img_izq = np.fliplr(img_izq).copy()
            img_der = np.fliplr(img_der).copy()
            label   = 1.0 - label

        par = np.concatenate([img_izq, img_der], axis=2).astype(np.float32) / 255.0
        par = (par - self.mean) / self.std
        return torch.from_numpy(par.transpose(2,0,1)), torch.tensor(label, dtype=torch.float32)


# Cargar splits de KITTI
with open(os.path.join(SPLITS_DIR, 'kitti_splits.json')) as f:
    kitti_splits = json.load(f)

train_ids = kitti_splits.get('train', kitti_splits.get('kitti_train', []))
test_ids  = kitti_splits.get('test',  kitti_splits.get('kitti_test',  []))

print(f'IDs train: {len(train_ids)} | IDs test: {len(test_ids)}')

ds_kitti_train = KITTIDataset(train_ids, KITTI_LEFT_DIR, KITTI_RIGHT_DIR, KITTI_DISP_DIR, augment=True)
ds_kitti_test  = KITTIDataset(test_ids,  KITTI_LEFT_DIR, KITTI_RIGHT_DIR, KITTI_DISP_DIR, augment=False)

## Celda 3 — Dataset de Ilusiones Visuales

In [ ]:
class IllusionDataset(Dataset):
    """
    Dataset 3D Visual Illusion Depth Estimation (Yao et al., NeurIPS 2025).
    Entrada: imagen duplicada en par estereoscópico (6 canales).
    Etiqueta: label_profundidad del metadata enriquecido (0 o 1).
    """
    def __init__(self, df_meta, images_dir, target_size=(192, 640), augment=False):
        self.images_dir  = images_dir
        self.target_size = target_size
        self.augment     = augment
        self.mean = np.array([0.485,0.456,0.406,0.485,0.456,0.406], dtype=np.float32)
        self.std  = np.array([0.229,0.224,0.225,0.229,0.224,0.225], dtype=np.float32)

        # Filtrar imágenes válidas con etiqueta
        self.items = []
        for _, row in df_meta.iterrows():
            label = row.get('label_profundidad', -1)
            if label not in [0, 1]:
                continue
            ruta = os.path.join(images_dir, row['filename'])
            if os.path.exists(ruta):
                self.items.append((ruta, float(label)))

        print(f'IllusionDataset: {len(self.items)} imágenes válidas')
        labels = [l for _, l in self.items]
        print(f'  Cercano (1): {sum(l==1 for l in labels)} | Lejano (0): {sum(l==0 for l in labels)}')

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        ruta, label = self.items[idx]
        img = cv2.imread(ruta)
        if img is None:
            img = np.zeros((192, 640, 3), dtype=np.uint8)

        h, w = self.target_size
        img  = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), (w, h))

        if self.augment:
            if np.random.rand() > 0.5:
                img   = np.fliplr(img).copy()
                label = 1.0 - label
            if np.random.rand() > 0.5:
                factor = np.random.uniform(0.8, 1.2)
                img    = np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)

        # Duplicar imagen (par estereoscópico simulado)
        par = np.concatenate([img, img], axis=2).astype(np.float32) / 255.0
        par = (par - self.mean) / self.std
        return torch.from_numpy(par.transpose(2,0,1)), torch.tensor(label, dtype=torch.float32)


# Cargar metadata de ilusiones
df_meta_ill = pd.read_csv(ILLUSION_META)
print(f'Metadata ilusiones: {len(df_meta_ill)} imágenes')
print(f'Columnas: {df_meta_ill.columns.tolist()}')
print()

# Split 80/20 para ilusiones
df_meta_ill = df_meta_ill.sample(frac=1, random_state=SEED).reset_index(drop=True)
n_train_ill = int(len(df_meta_ill) * 0.8)
df_ill_train = df_meta_ill.iloc[:n_train_ill]
df_ill_test  = df_meta_ill.iloc[n_train_ill:]

ds_ill_train = IllusionDataset(df_ill_train, ILLUSION_DIR, augment=True)
ds_ill_test  = IllusionDataset(df_ill_test,  ILLUSION_DIR, augment=False)

## Celda 4 — Combinar datasets y crear DataLoaders

In [ ]:
# Combinar KITTI + Ilusiones
ds_train_combined = ConcatDataset([ds_kitti_train, ds_ill_train])
ds_test_combined  = ConcatDataset([ds_kitti_test,  ds_ill_test])

BATCH_SIZE  = 8   # Reducido para caber en VRAM con imágenes 192×640×6
NUM_WORKERS = 2

train_loader = DataLoader(
    ds_train_combined,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True
)
test_loader = DataLoader(
    ds_test_combined,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print('Datasets combinados:')
print(f'  Train: {len(ds_train_combined)} muestras '
      f'({len(ds_kitti_train)} KITTI + {len(ds_ill_train)} ilusiones)')
print(f'  Test:  {len(ds_test_combined)} muestras '
      f'({len(ds_kitti_test)} KITTI + {len(ds_ill_test)} ilusiones)')
print(f'  Batch size: {BATCH_SIZE}')
print(f'  Batches por época: {len(train_loader)}')
print()

# Verificar un batch
x_batch, y_batch = next(iter(train_loader))
print(f'  Forma del batch: X={x_batch.shape}, y={y_batch.shape}')
print(f'  Labels en batch: {y_batch.tolist()}')

## Celda 5 — Definir arquitectura del modelo

In [ ]:
# Misma arquitectura que en NB04/NB05 original
# Reconstruida desde los shapes exactos del checkpoint best_model.pth

class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(ch)
        self.bn2   = nn.BatchNorm2d(ch)
        self.relu  = nn.ReLU(inplace=True)
    def forward(self, x):
        r = self.relu(self.bn1(self.conv1(x)))
        r = self.bn2(self.conv2(r))
        return self.relu(x + r)

class ConvBnSkip(nn.Module):
    def __init__(self, in_ch, out_ch, k=3):
        super().__init__()
        self.conv    = nn.Conv2d(in_ch, out_ch, k, padding=k//2)
        self.bn      = nn.BatchNorm2d(out_ch)
        self.skip    = nn.Conv2d(in_ch, out_ch, 1)
        self.skip_bn = nn.BatchNorm2d(out_ch)
        self.relu    = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(self.bn(self.conv(x))) + self.relu(self.skip_bn(self.skip(x)))

class NGL(nn.Module):
    def __init__(self):
        super().__init__()
        self.magno_pathway = nn.Sequential(
            nn.Conv2d(6,8,5,padding=2), nn.BatchNorm2d(8), nn.ReLU(inplace=True), ResBlock(8))
        self.parvo_pathway = nn.Sequential(
            nn.Conv2d(6,8,3,padding=1), nn.BatchNorm2d(8), nn.ReLU(inplace=True), ResBlock(8))
    def forward(self, x):
        return self.magno_pathway(x), self.parvo_pathway(x)

class V1(nn.Module):
    def __init__(self):
        super().__init__()
        self.iv_c_alpha     = nn.Sequential(nn.Conv2d(8,16,3,padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), ResBlock(16))
        self.iv_c_beta      = nn.Sequential(nn.Conv2d(8,16,3,padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), ResBlock(16))
        self.iv_b           = nn.Sequential(ResBlock(16), ResBlock(16))
        self.iv_a           = nn.Sequential(nn.Conv2d(32,16,1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), ResBlock(16))
        self.layers_ii_iii  = nn.Sequential(ResBlock(16), ResBlock(16))
        self.layer_v        = nn.Sequential(ResBlock(16))
        self.layer_vi_and_i = nn.Sequential(ResBlock(16))
    def forward(self, magno, parvo, feedback=None):
        alpha = self.iv_c_alpha(parvo)
        beta  = self.iv_c_beta(magno)
        _     = self.iv_b(beta)
        iva   = self.iv_a(torch.cat([alpha, beta], dim=1))
        if feedback is not None: iva = iva + feedback
        return self.layer_vi_and_i(self.layer_v(self.layers_ii_iii(iva)))

class V2(nn.Module):
    def __init__(self):
        super().__init__()
        self.thick_bands  = nn.Sequential(ConvBnSkip(16,32), ResBlock(32))
        self.thin_bands   = nn.Sequential(ConvBnSkip(16,32), ResBlock(32))
        self.interstripes = nn.Sequential(nn.Conv2d(64,32,1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), ResBlock(32))
    def forward(self, x, feedback=None):
        thick = self.thick_bands(x)
        thin  = self.thin_bands(x)
        inter = self.interstripes(torch.cat([thick, thin], dim=1))
        if feedback is not None: inter = inter + feedback
        return thick, thin, inter

class V3(nn.Module):
    def __init__(self):
        super().__init__()
        self.v3a_disparity = nn.Sequential(ConvBnSkip(32,64), ResBlock(64), ResBlock(64))
        self.v3a_motion    = nn.Sequential(ConvBnSkip(32,64), ResBlock(64))
        self.v3_shapes     = nn.Sequential(nn.Conv2d(128,64,1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), ResBlock(64), ResBlock(64))
    def forward(self, thick, thin):
        disp   = self.v3a_disparity(thick)
        motion = self.v3a_motion(thick)
        shapes = self.v3_shapes(torch.cat([disp, motion], dim=1))
        return disp, motion, shapes

class V4(nn.Module):
    def __init__(self):
        super().__init__()
        self.v4a_color      = nn.Sequential(ConvBnSkip(32,64), ResBlock(64))
        self.v4a_shapes     = nn.Sequential(ResBlock(64))
        self.v4b_attention  = nn.Sequential(nn.Conv2d(128,64,1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), ResBlock(64))
        self.attention_gate = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64,64), nn.Sigmoid())
    def forward(self, thin, shapes):
        attn = self.v4b_attention(torch.cat([self.v4a_color(thin), self.v4a_shapes(shapes)], dim=1))
        gate = self.attention_gate(attn).unsqueeze(-1).unsqueeze(-1)
        return attn * gate

class V5MT(nn.Module):
    def __init__(self):
        super().__init__()
        self.motion_analysis       = nn.Sequential(ConvBnSkip(32,128), ResBlock(128), ResBlock(128))
        self.disparity_integration = nn.Sequential(ConvBnSkip(64,128), ResBlock(128))
        self.dynamic_maps          = nn.Sequential(nn.Conv2d(256,128,1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), ResBlock(128), ResBlock(128), ResBlock(128))
    def forward(self, motion_in, disparity):
        return self.dynamic_maps(torch.cat([self.motion_analysis(motion_in), self.disparity_integration(disparity)], dim=1))

class FeedbackModule(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.transform = nn.Sequential(nn.Conv2d(in_ch,out_ch,1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.transform(x)

class OutputLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.integration_conv = nn.Sequential(nn.Conv2d(192,128,1), nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(128,256), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(256,64),  nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(64,1),    nn.Sigmoid())
    def forward(self, v4_out, v5mt_out):
        return self.classifier(self.integration_conv(torch.cat([v4_out, v5mt_out], dim=1)))

class CognitiveDepthModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.ngl               = NGL()
        self.v1                = V1()
        self.v2                = V2()
        self.v3                = V3()
        self.v4                = V4()
        self.v5mt              = V5MT()
        self.output_layer      = OutputLayer()
        self.feedback_v2_to_v1 = FeedbackModule(32,16)
        self.feedback_v3_to_v2 = FeedbackModule(64,32)
    def forward(self, x):
        magno, parvo         = self.ngl(x)
        v1_out               = self.v1(magno, parvo)
        thick, thin, inter   = self.v2(v1_out)
        v1_out               = self.v1(magno, parvo, feedback=self.feedback_v2_to_v1(inter))
        thick, thin, inter   = self.v2(v1_out)
        disp, motion, shapes = self.v3(thick, thin)
        thick, thin, inter   = self.v2(v1_out, feedback=self.feedback_v3_to_v2(disp))
        disp, motion, shapes = self.v3(thick, thin)
        v4_out               = self.v4(thin, shapes)
        v5mt_out             = self.v5mt(thick, disp)
        return self.output_layer(v4_out, v5mt_out)

model = CognitiveDepthModel().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'✓ Modelo cargado: {total_params:,} parámetros')

# Verificar forward pass
with torch.no_grad():
    test_out = model(torch.randn(2, 6, 192, 640).to(device))
    print(f'  Forward pass OK: {test_out.shape}')

## Celda 6 — Funciones de entrenamiento y evaluación

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_correct, total_n = 0.0, 0, 0
    all_preds, all_labels = [], []
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out  = model(X).squeeze()
        loss = criterion(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        preds = (out > 0.5).float()
        total_loss    += loss.item() * len(y)
        total_correct += (preds == y).sum().item()
        total_n       += len(y)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    f1  = f1_score(all_labels, all_preds, zero_division=0)
    return {'loss': total_loss/total_n, 'accuracy': total_correct/total_n, 'f1': f1}


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_n = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out  = model(X).squeeze()
            loss = criterion(out, y)
            preds = (out > 0.5).float()
            total_loss    += loss.item() * len(y)
            total_correct += (preds == y).sum().item()
            total_n       += len(y)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    f1  = f1_score(all_labels, all_preds, zero_division=0)
    return {'loss': total_loss/total_n, 'accuracy': total_correct/total_n, 'f1': f1}


class EarlyStopping:
    def __init__(self, patience=15, min_delta=0.001):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_loss  = float('inf')
        self.counter    = 0
        self.stop       = False
    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True


def freeze_ngl_v1(model):
    for param in model.ngl.parameters(): param.requires_grad = False
    for param in model.v1.parameters():  param.requires_grad = False
    print('  NGL y V1 congelados para Etapa 1.')

def unfreeze_all(model):
    for param in model.parameters(): param.requires_grad = True
    print('  Todos los parámetros descongelados para Etapa 2.')


print('✓ Funciones de entrenamiento definidas.')

## Celda 7 — Etapa 1: Entrenamiento con NGL+V1 congelados (30 épocas)

> **Tiempo estimado:** ~2-3 horas en Tesla T4

In [ ]:
# Inicializar modelo desde cero (reentrenamiento completo)
model = CognitiveDepthModel().to(device)
torch.manual_seed(SEED)

freeze_ngl_v1(model)

criterion   = nn.BCELoss()
optimizer   = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001, weight_decay=1e-4
)
scheduler   = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=10, factor=0.5, min_lr=1e-7
)
early_stop  = EarlyStopping(patience=15)

STAGE1_EPOCHS = 30
history_s1    = {'train': [], 'test': [], 'epoch': [], 'lr': []}
best_loss_s1  = float('inf')

print('=' * 60)
print('ETAPA 1: Entrenamiento con NGL+V1 congelados')
print(f'Épocas: {STAGE1_EPOCHS} | lr: 0.001 | batch: {BATCH_SIZE}')
print('=' * 60)

for epoch in range(STAGE1_EPOCHS):
    t0 = time.time()
    train_m = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_m  = evaluate(model, test_loader, criterion, device)
    lr      = optimizer.param_groups[0]['lr']
    scheduler.step(test_m['loss'])
    elapsed = time.time() - t0

    history_s1['train'].append(train_m)
    history_s1['test'].append(test_m)
    history_s1['epoch'].append(epoch + 1)
    history_s1['lr'].append(lr)

    print(f'E{epoch+1:02d}/{STAGE1_EPOCHS} | '
          f'Train loss={train_m["loss"]:.4f} acc={train_m["accuracy"]:.3f} f1={train_m["f1"]:.3f} | '
          f'Test  loss={test_m["loss"]:.4f} acc={test_m["accuracy"]:.3f} | '
          f'lr={lr:.2e} | {elapsed:.0f}s')

    # Guardar mejor modelo de Etapa 1
    if test_m['loss'] < best_loss_s1:
        best_loss_s1 = test_m['loss']
        torch.save({
            'epoch':            epoch + 1,
            'stage':            1,
            'model_state_dict': model.state_dict(),
            'test_loss':        test_m['loss'],
            'val_acc':          test_m['accuracy'],
            'metrics':          test_m
        }, os.path.join(CHECKPOINT_DIR, 'best_model_v2.pth'))

    early_stop(test_m['loss'])
    if early_stop.stop:
        print(f'  Early stopping en época {epoch+1}.')
        break

print()
print(f'✓ Etapa 1 completada. Mejor test loss: {best_loss_s1:.4f}')

## Celda 8 — Etapa 2: Fine-tuning completo (70 épocas)

> **Tiempo estimado:** ~5-6 horas en Tesla T4

In [ ]:
# Cargar mejor modelo de Etapa 1
checkpoint = torch.load(
    os.path.join(CHECKPOINT_DIR, 'best_model_v2.pth'),
    map_location=device, weights_only=False
)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Mejor modelo Etapa 1 cargado (época {checkpoint["epoch"]}, '
      f'test_loss={checkpoint["test_loss"]:.4f})')

unfreeze_all(model)

criterion   = nn.BCELoss()
optimizer   = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler   = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=10, factor=0.5, min_lr=1e-7
)
early_stop  = EarlyStopping(patience=15)

STAGE2_EPOCHS  = 70
history_s2     = {'train': [], 'test': [], 'epoch': [], 'lr': []}
best_loss_s2   = checkpoint['test_loss']

print()
print('=' * 60)
print('ETAPA 2: Fine-tuning completo del modelo')
print(f'Épocas: {STAGE2_EPOCHS} | lr: 0.0001 | batch: {BATCH_SIZE}')
print('=' * 60)

for epoch in range(STAGE2_EPOCHS):
    t0 = time.time()
    train_m = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_m  = evaluate(model, test_loader, criterion, device)
    lr      = optimizer.param_groups[0]['lr']
    scheduler.step(test_m['loss'])
    elapsed = time.time() - t0

    history_s2['train'].append(train_m)
    history_s2['test'].append(test_m)
    history_s2['epoch'].append(epoch + 1)
    history_s2['lr'].append(lr)

    print(f'E{epoch+1:02d}/{STAGE2_EPOCHS} | '
          f'Train loss={train_m["loss"]:.4f} acc={train_m["accuracy"]:.3f} f1={train_m["f1"]:.3f} | '
          f'Test  loss={test_m["loss"]:.4f} acc={test_m["accuracy"]:.3f} | '
          f'lr={lr:.2e} | {elapsed:.0f}s')

    if test_m['loss'] < best_loss_s2:
        best_loss_s2 = test_m['loss']
        torch.save({
            'epoch':            epoch + 1,
            'stage':            2,
            'model_state_dict': model.state_dict(),
            'test_loss':        test_m['loss'],
            'val_acc':          test_m['accuracy'],
            'metrics':          test_m
        }, os.path.join(CHECKPOINT_DIR, 'best_model_v2.pth'))
        print(f'  ✓ Nuevo mejor modelo guardado (test_loss={best_loss_s2:.4f})')

    early_stop(test_m['loss'])
    if early_stop.stop:
        print(f'  Early stopping en época {epoch+1}.')
        break

# Guardar modelo final
torch.save({
    'epoch':            epoch + 1,
    'stage':            2,
    'model_state_dict': model.state_dict(),
    'test_loss':        test_m['loss'],
    'val_acc':          test_m['accuracy'],
    'metrics':          test_m
}, os.path.join(CHECKPOINT_DIR, 'model_final_v2.pth'))

print()
print(f'✓ Etapa 2 completada. Mejor test loss: {best_loss_s2:.4f}')

## Celda 9 — Visualización del historial de entrenamiento

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Historial de Entrenamiento — NB05b (KITTI + Ilusiones)',
             fontsize=13, fontweight='bold')

e1 = history_s1['epoch']
e2 = [e + max(e1) for e in history_s2['epoch']]
epochs_all = e1 + e2

train_loss = [m['loss'] for m in history_s1['train']] + [m['loss'] for m in history_s2['train']]
test_loss  = [m['loss'] for m in history_s1['test']]  + [m['loss'] for m in history_s2['test']]
train_acc  = [m['accuracy'] for m in history_s1['train']] + [m['accuracy'] for m in history_s2['train']]
test_acc   = [m['accuracy'] for m in history_s1['test']]  + [m['accuracy'] for m in history_s2['test']]
train_f1   = [m['f1'] for m in history_s1['train']] + [m['f1'] for m in history_s2['train']]
lrs_all    = history_s1['lr'] + history_s2['lr']

sep = len(e1)  # línea divisoria entre etapas

for ax, y1, y2, ylabel, title in [
    (axes[0,0], train_loss, test_loss,  'Loss (BCE)',    'Función de Pérdida'),
    (axes[0,1], train_acc,  test_acc,   'Accuracy',      'Precisión'),
]:
    ax.plot(epochs_all, y1, 'b-', alpha=0.8, label='Train')
    ax.plot(epochs_all, y2, 'r-', alpha=0.8, label='Test')
    ax.axvline(epochs_all[sep-1], color='gray', ls='--', lw=1, label='Fin Etapa 1')
    ax.set_xlabel('Época'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(); ax.grid(True, alpha=0.3)

axes[1,0].plot(epochs_all, train_f1, 'g-', alpha=0.8, label='Train F1')
axes[1,0].axvline(epochs_all[sep-1], color='gray', ls='--', lw=1)
axes[1,0].set_xlabel('Época'); axes[1,0].set_ylabel('F1 Score')
axes[1,0].set_title('F1 Score (Train)'); axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

axes[1,1].semilogy(epochs_all, lrs_all, 'purple', alpha=0.8)
axes[1,1].axvline(epochs_all[sep-1], color='gray', ls='--', lw=1)
axes[1,1].set_xlabel('Época'); axes[1,1].set_ylabel('Learning Rate')
axes[1,1].set_title('Learning Rate'); axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
ruta_fig = os.path.join(RESULTS_DIR, 'visualizations', 'training_history_v2.png')
os.makedirs(os.path.dirname(ruta_fig), exist_ok=True)
plt.savefig(ruta_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada: {ruta_fig}')

## Celda 10 — Evaluación final y resumen

In [ ]:
# Cargar mejor modelo final
checkpoint = torch.load(
    os.path.join(CHECKPOINT_DIR, 'best_model_v2.pth'),
    map_location=device, weights_only=False
)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print('=' * 60)
print('EVALUACIÓN FINAL — MODELO v2 (KITTI + Ilusiones)')
print('=' * 60)
print(f'Mejor época: {checkpoint["epoch"]} | Etapa: {checkpoint["stage"]}')
print(f'Test loss:   {checkpoint["test_loss"]:.4f}')
print(f'Test acc:    {checkpoint["val_acc"]:.4f}  ({checkpoint["val_acc"]*100:.1f}%)')
print()

# Evaluación separada por dataset
kitti_loader = DataLoader(ds_kitti_test,  batch_size=BATCH_SIZE, shuffle=False)
ill_loader   = DataLoader(ds_ill_test,    batch_size=BATCH_SIZE, shuffle=False)

m_kitti = evaluate(model, kitti_loader, criterion, device)
m_ill   = evaluate(model, ill_loader,   criterion, device)

print('Desempeño por dataset:')
print(f'  KITTI:     acc={m_kitti["accuracy"]:.4f} ({m_kitti["accuracy"]*100:.1f}%) | f1={m_kitti["f1"]:.4f}')
print(f'  Ilusiones: acc={m_ill["accuracy"]:.4f}   ({m_ill["accuracy"]*100:.1f}%) | f1={m_ill["f1"]:.4f}')
print()

# Guardar historial CSV
history_df = pd.DataFrame({
    'epoch':         history_s1['epoch'] + [e + len(e1) for e in history_s2['epoch']],
    'stage':         [1]*len(history_s1['epoch']) + [2]*len(history_s2['epoch']),
    'lr':            history_s1['lr'] + history_s2['lr'],
    'train_loss':    [m['loss'] for m in history_s1['train']] + [m['loss'] for m in history_s2['train']],
    'train_accuracy':[m['accuracy'] for m in history_s1['train']] + [m['accuracy'] for m in history_s2['train']],
    'train_f1':      [m['f1'] for m in history_s1['train']] + [m['f1'] for m in history_s2['train']],
    'test_loss':     [m['loss'] for m in history_s1['test']] + [m['loss'] for m in history_s2['test']],
    'test_accuracy': [m['accuracy'] for m in history_s1['test']] + [m['accuracy'] for m in history_s2['test']],
})
ruta_hist = os.path.join(RESULTS_DIR, 'metrics', 'training_history_v2.csv')
os.makedirs(os.path.dirname(ruta_hist), exist_ok=True)
history_df.to_csv(ruta_hist, index=False)

print('Archivos generados:')
print(f'  checkpoints/best_model_v2.pth')
print(f'  checkpoints/model_final_v2.pth')
print(f'  results/metrics/training_history_v2.csv')
print(f'  results/visualizations/training_history_v2.png')
print()
print('Próximo paso → Actualizar NB10 para validar con best_model_v2.pth')